# Token-Level Linguistic Router — GPU run

Trains 20 per-language LoRA experts, runs the three-arm evaluation, and sweeps
the router config to a plateau.

**Works on Kaggle and Colab.** Before running:

- **Kaggle**: Settings → Accelerator → **GPU T4 x2**, and Settings →
  **Internet: On** (needed to pull the base model and benchmarks).
  **Do not pick P100** — it is sm_60 and Kaggle's PyTorch only ships sm_70+
  kernels, so it fails. Cell 1 checks this and stops early.
- **Colab**: Runtime → Change runtime type → **T4 GPU**.

Then Run All. Everything is self-contained — no git remote, no credentials.


## 1. Environment check

In [ ]:
import os, sys, subprocess

gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                      '--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip()
print(gpu or 'NO GPU DETECTED — enable the accelerator in settings, then rerun')

IN_KAGGLE = os.path.isdir('/kaggle')
IN_COLAB = os.path.isdir('/content') and not IN_KAGGLE
WORK = '/kaggle/working' if IN_KAGGLE else ('/content/work' if IN_COLAB else '.')
os.makedirs(WORK, exist_ok=True)
os.chdir(WORK)
print('platform:', 'kaggle' if IN_KAGGLE else 'colab' if IN_COLAB else 'local')
print('workdir :', os.getcwd())

# Fail here rather than 20 minutes in. Kaggle still offers P100 (sm_60), but
# its PyTorch build only ships kernels for sm_70+. Selecting P100 produces a
# pile of warnings and then dies on the first real kernel launch, which is a
# much more confusing failure than this assertion.
try:
    import torch
    if torch.cuda.is_available():
        major, minor = torch.cuda.get_device_capability(0)
        name = torch.cuda.get_device_name(0)
        arches = torch.cuda.get_arch_list()
        print(f'device  : {name} (sm_{major}{minor})')
        if f'sm_{major}{minor}' not in arches:
            raise SystemExit(
                f'\nINCOMPATIBLE GPU: {name} is sm_{major}{minor}, but this '
                f'PyTorch supports {arches}.\n'
                f'Fix: right sidebar -> Accelerator -> "GPU T4 x2" (T4 is '
                f'sm_75), then rerun.\n'
                f'P100 is sm_60 and will not work without rebuilding PyTorch.'
            )
    else:
        print('WARNING: CUDA not available — this will run on CPU and be very slow.')
except ImportError:
    pass   # torch arrives with the install cell below

## 2. Dependencies

In [ ]:
# Kaggle preinstalls torchao 0.10.0. PEFT's is_torchao_available() *raises*
# on a version below 0.16 instead of returning False, so building any LoRA
# module dies with "Found an incompatible version of torchao". Nothing here
# uses torchao, and with it absent that check returns False cleanly — so the
# fix is to remove it rather than chase a compatible build.
%pip uninstall -q -y torchao
%pip install -q -U transformers peft datasets accelerate safetensors langid pytest

import importlib.util
import torch, transformers, peft
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
print('bf16 supported:', torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False)
print('transformers', transformers.__version__, '| peft', peft.__version__)
print('torchao present:', importlib.util.find_spec('torchao') is not None, '(should be False)')

# Prove the LoRA path actually builds before spending 10 minutes on training.
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model
_m = get_peft_model(
    AutoModelForCausalLM.from_pretrained('sshleifer/tiny-gpt2', dtype=torch.float32),
    LoraConfig(r=4, target_modules=['c_attn'], task_type='CAUSAL_LM'),
    adapter_name='_smoke')
print('LoRA construction OK')

## 3. Write the source

Embedded verbatim from the repo by `notebooks/build_notebook.py`.

In [ ]:
import os, json

FILES = json.loads(r'''{"requirements.txt": "torch\ntransformers\naccelerate\npeft\ndatasets\nhuggingface_hub\nsentencepiece\nlangid\n", "router/__init__.py": "", "router/config.py": "\"\"\"Router hyperparameters in one place, so the optimization loop can sweep\nthem without editing module constants.\"\"\"\nfrom dataclasses import dataclass, asdict\n\n\n@dataclass(frozen=True)\nclass RouterConfig:\n    # How often (in generated tokens) the language signal is evaluated.\n    # Lower = more responsive, more detector calls.\n    check_every: int = 4\n\n    # Trailing decoded characters the detector sees. Too short starves the\n    # Latin classifier; too long makes the router laggy after a real shift.\n    window_chars: int = 64\n\n    # Minimum script-bearing characters before any verdict is offered.\n    min_chars: int = 4\n\n    # Latin-script languages are not separable by script, so they need more\n    # evidence and a confidence floor before a switch is allowed.\n    latin_min_chars: int = 40\n    latin_min_conf: float = 0.90\n\n    # Require the same new language to be observed on this many consecutive\n    # checks before actually swapping. Suppresses one-window flickers.\n    switch_patience: int = 1\n\n    # Stop as soon as the answer line is emitted.\n    stop_on_answer: bool = True\n\n    # MILU/MMMLU items need real reasoning before the answer line. At 200 the\n    # model was still mid-CoT when it hit the cap, so extraction returned None\n    # and every arm scored ~0 — a meaningless comparison. 512 lets most items\n    # actually reach \"Answer: <letter>\".\n    max_new_tokens: int = 512\n\n    # When an expert switch happens, the keys/values already in the cache were\n    # computed by the *previous* expert, so the new expert attends over a\n    # history it did not encode. Setting this rebuilds the cache under the new\n    # weights instead — exact, but it re-pays prefill on every switch, which\n    # is precisely the cost the unified design exists to avoid. Off by default;\n    # the eval turns it on for one arm to price the shortcut.\n    recompute_on_switch: bool = False\n\n    def to_dict(self):\n        return asdict(self)\n\n\nDEFAULT = RouterConfig()\n", "router/script_detect.py": "\"\"\"Language-shift signal detection over a rolling window of generated text.\n\nDesign: Unicode script is the primary signal — it is deterministic, costs no\nmodel call, and for most of the target set it is decisive on its own. Two\nscript families need extra work:\n\n  * CJK: Japanese mixes Han with Kana, and a Japanese sentence is often\n    *majority* Han, so plain \"dominant script wins\" would misroute it to\n    Chinese. Presence of Kana (or Hangul) is therefore treated as decisive\n    regardless of how much Han sits alongside it.\n  * Latin: en/de/fr/es/it/pt all occupy the same block, so script carries no\n    information. Those fall through to a statistical classifier (langid),\n    which needs a longer window and a confidence floor to be usable.\n\nAnything the detector is not confident about returns None, which callers read\nas \"keep the current expert\" rather than \"switch to a default\".\n\"\"\"\nfrom langid.langid import LanguageIdentifier, model as _langid_model\n\n# --- language registry -------------------------------------------------\n# Indic set comes from MILU (ai4bharat), the rest from MMMLU (openai).\n# `name` is the dataset config key, `code` is what adapter files are keyed on.\n\nINDIC_LANGS = {\n    \"Bengali\": \"bn\", \"Gujarati\": \"gu\", \"Hindi\": \"hi\", \"Kannada\": \"kn\",\n    \"Malayalam\": \"ml\", \"Marathi\": \"mr\", \"Odia\": \"or\", \"Punjabi\": \"pa\",\n    \"Tamil\": \"ta\", \"Telugu\": \"te\", \"English\": \"en\",\n}\n\n# MMMLU config name -> short code\nMMMLU_LANGS = {\n    \"ZH_CN\": \"zh\", \"JA_JP\": \"ja\", \"KO_KR\": \"ko\", \"DE_DE\": \"de\",\n    \"FR_FR\": \"fr\", \"ES_LA\": \"es\", \"IT_IT\": \"it\", \"PT_BR\": \"pt\",\n    \"AR_XY\": \"ar\",\n}\n\nLANG_CODE = dict(INDIC_LANGS)\n\n# Languages that share the Latin block and must be separated statistically.\nLATIN_LANGS = [\"en\", \"de\", \"fr\", \"es\", \"it\", \"pt\"]\n# Languages that share the Devanagari block.\nDEVANAGARI_LANGS = [\"hi\", \"mr\"]\n\n# (start, end) inclusive Unicode codepoint ranges per script block.\nSCRIPT_RANGES = {\n    \"bn\": [(0x0980, 0x09FF)],           # Bengali\n    \"gu\": [(0x0A80, 0x0AFF)],           # Gujarati\n    \"kn\": [(0x0C80, 0x0CFF)],           # Kannada\n    \"ml\": [(0x0D00, 0x0D7F)],           # Malayalam\n    \"or\": [(0x0B00, 0x0B7F)],           # Odia\n    \"pa\": [(0x0A00, 0x0A7F)],           # Gurmukhi / Punjabi\n    \"ta\": [(0x0B80, 0x0BFF)],           # Tamil\n    \"te\": [(0x0C00, 0x0C7F)],           # Telugu\n    \"ar\": [(0x0600, 0x06FF), (0x0750, 0x077F)],   # Arabic\n    \"ru\": [(0x0400, 0x04FF)],           # Cyrillic\n    \"devanagari\": [(0x0900, 0x097F)],   # Hindi + Marathi\n    \"han\": [(0x4E00, 0x9FFF), (0x3400, 0x4DBF)],  # Chinese, also used in ja/ko\n    \"kana\": [(0x3040, 0x309F), (0x30A0, 0x30FF)],  # Hiragana + Katakana -> ja\n    \"hangul\": [(0xAC00, 0xD7AF), (0x1100, 0x11FF), (0x3130, 0x318F)],  # ko\n    \"latin\": [(0x0041, 0x005A), (0x0061, 0x007A), (0x00C0, 0x024F)],\n}\n\n# Independent classifiers so the two ambiguous families don't clobber each\n# other's language restriction (langid's restriction is global per-instance).\n_latin_id = LanguageIdentifier.from_modelstring(_langid_model, norm_probs=True)\n_latin_id.set_languages(LATIN_LANGS)\n\n_deva_id = LanguageIdentifier.from_modelstring(_langid_model, norm_probs=True)\n_deva_id.set_languages(DEVANAGARI_LANGS)\n\n# langid separates hi/mr poorly (it inverts the pair on short text), so that\n# one case uses closed-class function words instead — copulas, conjunctions,\n# negators and genitive markers, which differ sharply between the two and are\n# frequent enough to show up in a short window.\nHINDI_CUES = {\n    \"है\", \"हैं\", \"और\", \"मैं\", \"नहीं\", \"का\", \"की\", \"के\", \"यह\", \"वह\",\n    \"को\", \"से\", \"में\", \"कि\", \"था\", \"थे\", \"थी\", \"हुआ\", \"करना\", \"गया\",\n}\nMARATHI_CUES = {\n    \"आहे\", \"आहेत\", \"आणि\", \"मी\", \"नाही\", \"चा\", \"ची\", \"चे\", \"हे\", \"ते\",\n    \"मला\", \"त्या\", \"होते\", \"केले\", \"या\", \"व\", \"असे\", \"काय\", \"पण\",\n}\n# U+0933 (ळ) is standard in Marathi and effectively absent from Hindi.\nMARATHI_CHAR = \"ळ\"\n\n_DEVA_PUNCT = \"।,.?!\\\"'()[]{}:;०१२३४५६७८९\"\n\n\ndef _disambiguate_devanagari(text):\n    tokens = [t.strip(_DEVA_PUNCT) for t in text.split()]\n    hi_score = sum(1 for t in tokens if t in HINDI_CUES)\n    mr_score = sum(1 for t in tokens if t in MARATHI_CUES)\n    if MARATHI_CHAR in text:\n        mr_score += 2\n    if hi_score != mr_score:\n        return \"hi\" if hi_score > mr_score else \"mr\"\n    # No lexical evidence either way — fall back to the statistical model, and\n    # if that is also unsure, report undecided rather than guessing.\n    lang, conf = _deva_id.classify(text)\n    if conf >= 0.95 and lang in DEVANAGARI_LANGS:\n        return lang\n    return None\n\n# Latin needs far more evidence than an alphabet-unique script does.\nLATIN_MIN_CHARS = 40\nLATIN_MIN_CONF = 0.90\n\n\ndef _char_script(ch):\n    cp = ord(ch)\n    for lang, ranges in SCRIPT_RANGES.items():\n        for lo, hi in ranges:\n            if lo <= cp <= hi:\n                return lang\n    return None\n\n\ndef script_counts(text):\n    counts = {}\n    for ch in text:\n        s = _char_script(ch)\n        if s is not None:\n            counts[s] = counts.get(s, 0) + 1\n    return counts\n\n\ndef detect_language(text, min_chars=4, latin_min_chars=LATIN_MIN_CHARS,\n                    latin_min_conf=LATIN_MIN_CONF):\n    \"\"\"Return a short language code for `text`, or None if undecidable.\n\n    None means \"not enough evidence\" — callers keep the active expert instead\n    of falling back to a default, so the router doesn't thrash on whitespace,\n    digits, or a few shared punctuation characters.\n    \"\"\"\n    counts = script_counts(text)\n    total = sum(counts.values())\n    if total < min_chars:\n        return None\n\n    # CJK disambiguation runs before the generic dominant-script rule:\n    # Kana and Hangul are exclusive to Japanese and Korean respectively, and\n    # their presence outweighs any amount of co-occurring Han.\n    if counts.get(\"kana\", 0) > 0:\n        return \"ja\"\n    if counts.get(\"hangul\", 0) > 0:\n        return \"ko\"\n\n    dominant = max(counts, key=counts.get)\n\n    if dominant == \"han\":\n        return \"zh\"\n\n    if dominant == \"devanagari\":\n        return _disambiguate_devanagari(text)\n\n    if dominant == \"latin\":\n        # Script carries no signal here; require a longer window and a\n        # confident classifier verdict, else report \"undecided\".\n        if counts[\"latin\"] < latin_min_chars:\n            return None\n        lang, conf = _latin_id.classify(text)\n        if conf < latin_min_conf:\n            return None\n        return lang\n\n    return dominant\n\n\ndef language_shifted(prev_lang, window_text, cfg=None):\n    \"\"\"Router decision: return the new language code if the active-expert\n    language should change, else None.\"\"\"\n    from .config import DEFAULT\n    cfg = cfg or DEFAULT\n    new_lang = detect_language(\n        window_text,\n        min_chars=cfg.min_chars,\n        latin_min_chars=cfg.latin_min_chars,\n        latin_min_conf=cfg.latin_min_conf,\n    )\n    if new_lang is None or new_lang == prev_lang:\n        return None\n    return new_lang\n", "router/model_manager.py": "\"\"\"Shared-base + hot-swappable LoRA experts.\n\nThe trick that makes \"switch model weights mid-generation\" compatible with\n\"maintain a single continuous token history\": only the LoRA deltas move, the\nfrozen base transformer (and therefore every cached key/value already\ncomputed) never changes. Swapping the active adapter is a dict-pointer flip\n(`PeftModel.set_adapter`), not a reload, so `past_key_values` computed under\none expert stays valid input to the next.\n\"\"\"\nimport os\nimport torch\nfrom safetensors.torch import load_file, save_file\nfrom transformers import AutoModelForCausalLM, AutoTokenizer\nfrom peft import (LoraConfig, PeftModel, get_peft_model,\n                  get_peft_model_state_dict, set_peft_model_state_dict)\n\nBASE_MODEL = \"Qwen/Qwen2.5-1.5B-Instruct\"\nADAPTER_DIR = os.path.join(os.path.dirname(__file__), \"..\", \"adapters\")\n\n\ndef _device():\n    if torch.cuda.is_available():\n        return \"cuda\"\n    if torch.backends.mps.is_available():\n        return \"mps\"\n    return \"cpu\"\n\n\ndef _dtype(device):\n    \"\"\"Half precision on CUDA, fp32 elsewhere.\n\n    bf16 where supported (Ampere+). On Turing (T4, sm_75) bf16 is unavailable\n    and falling back to fp32 costs roughly 8x — T4 does ~8 TFLOPS fp32 against\n    ~65 TFLOPS fp16 — which turns a 30-minute evaluation into an overnight one.\n    So T4 uses fp16.\n\n    MPS stays fp32 deliberately: half precision there has been a source of\n    silent numerical drift, and this experiment turns on comparing logits\n    across an adapter swap.\n\n    Override with WSC_DTYPE=float32|float16|bfloat16 if a run looks numerically\n    suspect.\n    \"\"\"\n    override = os.environ.get(\"WSC_DTYPE\")\n    if override:\n        return getattr(torch, override)\n    if device == \"cuda\":\n        return torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16\n    return torch.float32\n\n\nclass ExpertManager:\n    \"\"\"Owns one base model and a registry of per-language LoRA adapters.\"\"\"\n\n    def __init__(self, base_model=BASE_MODEL, adapter_dir=ADAPTER_DIR,\n                 device=None, dtype=None):\n        self.device = device or _device()\n        self.dtype = dtype or _dtype(self.device)\n        self.tokenizer = AutoTokenizer.from_pretrained(base_model)\n        base = AutoModelForCausalLM.from_pretrained(\n            base_model, dtype=self.dtype\n        ).to(self.device)\n        base.eval()\n\n        # Seed with a throwaway adapter so the model is a PeftModel from the\n        # start; every real language adapter gets added alongside it.\n        #\n        # `embed_tokens` is in the target set deliberately. Adapting only\n        # q_proj/v_proj leaves the token embeddings frozen and shared, so an\n        # \"expert\" would have no language-specific representation of tokens at\n        # all. LoRA on the embedding gives each language its own embedding\n        # delta (~1.2M params at r=8) while keeping the tokenizer and the\n        # cache geometry identical — which is what lets the KV cache still be\n        # shared across a switch. Swapping in a genuinely different model per\n        # language would give better embeddings but different tokenization and\n        # different cache shapes, which would forbid cache reuse entirely.\n        # This model ties embed_tokens to lm_head, so tying the adapter too\n        # makes the language-specific embedding delta apply on the output side\n        # as well — the expert scores tokens in its language differently, not\n        # just reads them differently.\n        seed_cfg = LoraConfig(\n            r=8, lora_alpha=16, lora_dropout=0.0,\n            target_modules=[\"q_proj\", \"v_proj\", \"embed_tokens\"],\n            task_type=\"CAUSAL_LM\",\n            ensure_weight_tying=True,\n        )\n        self.model = get_peft_model(base, seed_cfg, adapter_name=\"_base\")\n        self.model.eval()\n        self.loaded = {\"_base\"}\n        self.active = \"_base\"\n\n        self.seed_cfg = seed_cfg\n        self.adapter_dir = adapter_dir\n        if os.path.isdir(adapter_dir):\n            for lang in sorted(os.listdir(adapter_dir)):\n                path = os.path.join(adapter_dir, lang, \"lora.safetensors\")\n                if os.path.isfile(path):\n                    self.load_expert(lang, path)\n\n    def save_expert(self, code, path=None):\n        \"\"\"Persist only the LoRA tensors.\n\n        `save_pretrained` would also write the tied base layers — with\n        `ensure_weight_tying` that means the full 233M embed_tokens *and* the\n        233M lm_head alongside a 2.3M delta, i.e. ~1.7GB per language. Those\n        base weights are identical for every expert and already on disk with\n        the base model, so only the delta is stored.\n        \"\"\"\n        path = path or os.path.join(self.adapter_dir, code, \"lora.safetensors\")\n        os.makedirs(os.path.dirname(path), exist_ok=True)\n        state = get_peft_model_state_dict(self.model, adapter_name=code)\n        lora_only = {k: v.detach().cpu().contiguous()\n                     for k, v in state.items() if \"lora_\" in k}\n        save_file(lora_only, path)\n        return path\n\n    def load_expert(self, code, path=None):\n        path = path or os.path.join(self.adapter_dir, code, \"lora.safetensors\")\n        if code not in self.model.peft_config:\n            self.model.add_adapter(code, self.seed_cfg)\n        state = load_file(path)\n        set_peft_model_state_dict(self.model, state, adapter_name=code)\n        self.loaded.add(code)\n        return code\n\n    def available_experts(self):\n        return sorted(self.loaded - {\"_base\"})\n\n    def set_active(self, lang):\n        \"\"\"Hot-swap the active expert. No-op (and no cache impact) if `lang`\n        has no trained adapter yet — falls back to the unmodified base.\"\"\"\n        target = lang if lang in self.loaded else \"_base\"\n        if target != self.active:\n            self.model.set_adapter(target)\n            self.active = target\n        return self.active\n", "router/generation.py": "\"\"\"Autoregressive loop over a single continuous token history.\n\nThe architectural claim being measured here is about *prefill*, not about\ngenerated tokens. Two ways to use language-specific experts:\n\n  dispatch    detect the language, hand the conversation to a separate\n              specialist model, and re-encode the whole context so that model\n              can see it. Every switch re-pays prefill over the full prefix.\n\n  unified     keep one token history and one KV cache, and swap only the LoRA\n              delta in place. The frozen base never changes, so keys/values\n              already computed stay valid and prefill is paid exactly once.\n\n`generate_unified` runs the second and additionally accounts for what the\nfirst would have cost on the same trajectory, so the saving is measured rather\nthan asserted.\n\"\"\"\nimport re\nimport torch\n\nfrom .config import DEFAULT\nfrom .script_detect import detect_language, language_shifted\n\nANSWER_RE = re.compile(r\"answer\\s*[:\\-]?\\s*\\(?([A-D])\\)?\", re.IGNORECASE)\n\n\ndef build_prompt(tokenizer, question, options):\n    opt_lines = \"\\n\".join(f\"{chr(65 + i)}. {o}\" for i, o in enumerate(options))\n    user = (\n        f\"{question}\\n{opt_lines}\\n\\n\"\n        \"Think step by step, then on the final line write exactly \"\n        \"'Answer: <letter>'.\"\n    )\n    messages = [{\"role\": \"user\", \"content\": user}]\n    return tokenizer.apply_chat_template(\n        messages, tokenize=False, add_generation_prompt=True\n    )\n\n\ndef generate_unified(manager, prompt, cfg=None, route=True, seed_text=None):\n    \"\"\"Greedy decode. With `route` set, the language-shift signal hot-swaps the\n    active expert mid-generation; with it unset the identical loop stays pinned\n    to the base weights (the baseline arm).\n\n    `seed_text` picks the *initial* expert. It matters: the full prompt is\n    mostly chat-template markup and an English instruction, so detecting on it\n    routes on the scaffolding rather than on the question — a Gujarati item\n    was landing on the French expert. Callers pass the raw question.\n    \"\"\"\n    cfg = cfg or DEFAULT\n    tok = manager.tokenizer\n    device = manager.device\n\n    eos_ids = set(tok.eos_token_id if isinstance(tok.eos_token_id, list)\n                  else [tok.eos_token_id])\n    im_end = tok.convert_tokens_to_ids(\"<|im_end|>\")\n    if im_end is not None and im_end >= 0:\n        eos_ids.add(im_end)\n\n    input_ids = tok(prompt, return_tensors=\"pt\").input_ids.to(device)\n    prompt_len = input_ids.shape[1]\n\n    if route:\n        manager.set_active(detect_language(seed_text or prompt) or \"_base\")\n    else:\n        manager.set_active(\"_base\")\n    cur_lang = manager.active\n\n    generated_ids = []\n    text_buffer = \"\"\n    switch_log = []\n    pending_lang, pending_count = None, 0\n    recompute_events = 0\n\n    # Prefill cost actually paid: the prompt, once.\n    prefill_tokens = prompt_len\n    # What a dispatch architecture would have paid: the prompt once, plus a\n    # re-encode of everything generated so far at each switch.\n    dispatch_prefill_tokens = prompt_len\n\n    with torch.no_grad():\n        out = manager.model(input_ids=input_ids, use_cache=True)\n        past = out.past_key_values\n        next_token_logits = out.logits[:, -1, :]\n\n        for step in range(cfg.max_new_tokens):\n            next_id = int(torch.argmax(next_token_logits, dim=-1).item())\n            if next_id in eos_ids:\n                break\n\n            generated_ids.append(next_id)\n            text_buffer += tok.decode([next_id])\n\n            if cfg.stop_on_answer and ANSWER_RE.search(text_buffer):\n                break\n\n            if route and (step + 1) % cfg.check_every == 0:\n                window = text_buffer[-cfg.window_chars:]\n                shift = language_shifted(cur_lang, window, cfg)\n\n                if shift is None:\n                    pending_lang, pending_count = None, 0\n                else:\n                    # Require the same verdict on consecutive checks before\n                    # committing, so a single noisy window can't cause a swap.\n                    if shift == pending_lang:\n                        pending_count += 1\n                    else:\n                        pending_lang, pending_count = shift, 1\n\n                    if pending_count >= cfg.switch_patience:\n                        switch_log.append(\n                            {\"step\": step, \"from\": cur_lang, \"to\": shift}\n                        )\n                        manager.set_active(shift)\n                        cur_lang = manager.active\n                        pending_lang, pending_count = None, 0\n                        # A dispatch design would re-encode the whole prefix\n                        # here; the unified design reuses the live cache.\n                        dispatch_prefill_tokens += prompt_len + len(generated_ids)\n\n                        if cfg.recompute_on_switch:\n                            # Rebuild the cache under the new expert so it\n                            # attends over a history it encoded itself. Exact,\n                            # but pays the re-prefill the unified design avoids.\n                            full = torch.cat(\n                                [input_ids,\n                                 torch.tensor([generated_ids], device=device)],\n                                dim=1,\n                            )\n                            out = manager.model(input_ids=full, use_cache=True)\n                            past = out.past_key_values\n                            next_token_logits = out.logits[:, -1, :]\n                            prefill_tokens += full.shape[1]\n                            recompute_events += 1\n                            continue\n\n            out = manager.model(\n                input_ids=torch.tensor([[next_id]], device=device),\n                past_key_values=past, use_cache=True,\n            )\n            past = out.past_key_values\n            next_token_logits = out.logits[:, -1, :]\n\n    n_gen = len(generated_ids)\n    return {\n        \"text\": tok.decode(generated_ids),\n        \"num_tokens\": n_gen,\n        \"prompt_tokens\": prompt_len,\n        \"prefill_tokens\": prefill_tokens,\n        \"dispatch_prefill_tokens\": dispatch_prefill_tokens,\n        \"prefill_saved\": dispatch_prefill_tokens - prefill_tokens,\n        \"total_processed\": prefill_tokens + n_gen,\n        \"dispatch_total_processed\": dispatch_prefill_tokens + n_gen,\n        \"switches\": switch_log,\n        \"recompute_events\": recompute_events,\n        \"final_lang\": cur_lang,\n    }\n\n\ndef extract_answer(text):\n    m = ANSWER_RE.search(text)\n    return m.group(1).upper() if m else None\n", "router/benchmarks.py": "\"\"\"Unified MCQ loader over two benchmarks, normalized to one row schema.\n\n  * MILU (ai4bharat) — 11 Indic languages. The canonical repo is gated; the\n    `murthyrudra/milu-cleaned` mirror carries the same per-language configs\n    and identical fields, so it is used by default with the gated original\n    tried first (so this transparently upgrades if access is granted).\n  * MMMLU (openai) — the non-Indic set: zh, ja, ko, de, fr, es, it, pt, ar.\n\nBoth are 4-option multiple choice, so they normalize cleanly onto:\n    {question, options[4], answer (letter A-D), lang (code), lang_name,\n     subject, source}\n\"\"\"\nimport json\nimport os\n\nfrom .script_detect import INDIC_LANGS, MMMLU_LANGS\n\nMILU_PRIMARY = \"ai4bharat/MILU\"\nMILU_MIRROR = \"murthyrudra/milu-cleaned\"\nMMMLU_REPO = \"openai/MMMLU\"\n\nFIXTURE_PATH = os.path.join(os.path.dirname(__file__), \"..\", \"data\", \"milu_stub.jsonl\")\n\n# Rows [0, EVAL_RESERVE) of a single-split benchmark are reserved for eval;\n# adapter training may only draw from beyond it.\nEVAL_RESERVE = 100\n\n# Every language the router can be evaluated on, in one table.\nALL_LANGS = [\n    # (dataset config name, short code, benchmark)\n    *[(name, code, \"milu\") for name, code in INDIC_LANGS.items()],\n    *[(name, code, \"mmmlu\") for name, code in MMMLU_LANGS.items()],\n]\n\n\ndef _norm_milu(row, lang_name, code):\n    \"\"\"MILU stores the answer as the literal string 'option3'.\"\"\"\n    options = [row[\"option1\"], row[\"option2\"], row[\"option3\"], row[\"option4\"]]\n    target = str(row[\"target\"]).strip()\n    if target.startswith(\"option\"):\n        idx = int(target.replace(\"option\", \"\")) - 1\n    else:\n        idx = int(target) - 1\n    return {\n        \"question\": row[\"question\"],\n        \"options\": options,\n        \"answer\": \"ABCD\"[idx],\n        \"lang\": code,\n        \"lang_name\": lang_name,\n        \"subject\": row.get(\"subject\", \"\"),\n        \"source\": \"milu\",\n    }\n\n\ndef _norm_mmmlu(row, lang_name, code):\n    \"\"\"MMMLU stores options in columns A-D and the answer as a letter.\"\"\"\n    return {\n        \"question\": row[\"Question\"],\n        \"options\": [row[\"A\"], row[\"B\"], row[\"C\"], row[\"D\"]],\n        \"answer\": str(row[\"Answer\"]).strip().upper(),\n        \"lang\": code,\n        \"lang_name\": lang_name,\n        \"subject\": row.get(\"Subject\", \"\"),\n        \"source\": \"mmmlu\",\n    }\n\n\ndef _load_hf(repo, config, split, limit):\n    from datasets import load_dataset\n    return load_dataset(repo, config, split=f\"{split}[:{limit}]\")\n\n\ndef _load_milu(lang_name, code, split, limit):\n    # MILU ships `validation` and `test`. Eval reads validation; adapter\n    # training reads test, so the two are disjoint by construction.\n    hf_split = \"validation\" if split == \"validation\" else \"test\"\n    for repo in (MILU_PRIMARY, MILU_MIRROR):\n        try:\n            ds = _load_hf(repo, lang_name, hf_split, limit)\n            return [_norm_milu(r, lang_name, code) for r in ds], repo\n        except Exception:\n            continue\n    return _fixture_rows(lang_name, code, split, limit), \"fixture\"\n\n\ndef _load_mmmlu(lang_name, code, split, limit):\n    # MMMLU ships only `test`, so train is carved out of a region past a fixed\n    # reserve held for eval — disjoint regardless of the two limit values.\n    from datasets import load_dataset\n    if split == \"validation\":\n        span = f\"test[:{limit}]\"\n    else:\n        span = f\"test[{EVAL_RESERVE}:{EVAL_RESERVE + limit}]\"\n    try:\n        ds = load_dataset(MMMLU_REPO, lang_name, split=span)\n        return [_norm_mmmlu(r, lang_name, code) for r in ds], MMMLU_REPO\n    except Exception:\n        return [], \"unavailable\"\n\n\ndef _fixture_rows(lang_name, code, split, limit):\n    \"\"\"Offline fallback used only if both MILU sources are unreachable.\"\"\"\n    if not os.path.isfile(FIXTURE_PATH):\n        return []\n    rows = []\n    with open(FIXTURE_PATH) as f:\n        for line in f:\n            r = json.loads(line)\n            if r[\"language\"] == lang_name and r.get(\"split\", \"validation\") == split:\n                rows.append(_norm_milu(r, lang_name, code))\n    return rows[:limit]\n\n\ndef load_language(lang_name, code, benchmark, split=\"validation\", limit=10):\n    if benchmark == \"milu\":\n        return _load_milu(lang_name, code, split, limit)\n    return _load_mmmlu(lang_name, code, split, limit)\n\n\ndef load_eval_set(limit_per_lang=5, split=\"validation\", langs=None):\n    \"\"\"Return (rows, sources) across every configured language.\"\"\"\n    selected = langs or ALL_LANGS\n    rows, sources = [], {}\n    for lang_name, code, benchmark in selected:\n        got, src = load_language(lang_name, code, benchmark, split, limit_per_lang)\n        rows.extend(got)\n        sources[code] = src\n    return rows, sources\n", "router/train_experts.py": "\"\"\"Fine-tune one small LoRA adapter per language.\n\nObjective: plain causal language modelling over text *in that language*.\n\nThis is deliberately not answer-format supervision. Training on targets like\n\"The answer is B.\" would teach the adapters to skip reasoning, and the eval\nprompt asks for step-by-step CoT — so the adapters would suppress the very\nbehaviour being measured, deflating generated-token counts in a way that\nlooks like a token-efficiency win but is really just truncated reasoning.\n\nTraining on in-language text instead makes each adapter a *language*\nspecialist rather than a *task* specialist, which is what the routing\narchitecture actually claims to exploit, and leaves the CoT format entirely\nto the shared base.\n\"\"\"\nimport os\nimport torch\nfrom torch.optim import AdamW\n\nfrom .benchmarks import ALL_LANGS, load_language\nfrom .model_manager import ExpertManager\n\n\ndef _language_text(row):\n    \"\"\"Question and options as a plain in-language passage. No prompt\n    scaffolding, no answer marker — just the language itself.\"\"\"\n    return row[\"question\"] + \"\\n\" + \"\\n\".join(row[\"options\"])\n\n\ndef _batch(tokenizer, rows, device, max_len=256):\n    out = []\n    for row in rows:\n        ids = tokenizer(_language_text(row), return_tensors=\"pt\",\n                        truncation=True, max_length=max_len).input_ids\n        if ids.shape[1] < 8:      # too short to carry signal\n            continue\n        out.append(ids.to(device))\n    return out\n\n\ndef train_one_language(manager, code, rows, steps=60, lr=1e-5):\n    # lr matters more than it looks. At 1e-4 for 60 steps the adapters destroy\n    # the base model's instruction-following: routed generations collapsed to a\n    # bare \"Answer: C\" in 3 tokens while the baseline still produced full CoT.\n    # That reads as a ~98% token saving but is really just a broken model.\n    # 1e-5 leaves the chat/CoT behaviour intact — verified before training.\n    model = manager.model\n\n    if code not in model.peft_config:\n        model.add_adapter(code, model.peft_config[\"_base\"])\n    model.set_adapter(code)\n    manager.active = code\n    model.train()\n\n    examples = _batch(manager.tokenizer, rows, manager.device)\n    if not examples:\n        return None, []\n\n    trainable = [p for p in model.parameters() if p.requires_grad]\n    opt = AdamW(trainable, lr=lr)\n\n    losses = []\n    for step in range(steps):\n        ids = examples[step % len(examples)]\n        out = model(input_ids=ids, labels=ids)   # standard LM loss\n        out.loss.backward()\n        opt.step()\n        opt.zero_grad()\n        losses.append(out.loss.detach().item())\n\n    model.eval()\n    save_path = manager.save_expert(code)\n    manager.loaded.add(code)\n    return save_path, losses\n\n\ndef train_all(steps=60, n_train=48, force=False):\n    \"\"\"Resumable: languages whose adapter is already on disk are skipped, so\n    an interrupted run continues by simply being restarted. Pass force=True to\n    retrain everything from scratch.\"\"\"\n    manager = ExpertManager()\n    trained = {}\n    for lang_name, code, benchmark in ALL_LANGS:\n        existing = os.path.join(manager.adapter_dir, code, \"lora.safetensors\")\n        if os.path.isfile(existing) and not force:\n            print(f\"{code:3s} ({lang_name:10s}) already trained — skipping\", flush=True)\n            continue\n\n        rows, source = load_language(lang_name, code, benchmark,\n                                     split=\"train\", limit=n_train)\n        if len(rows) < 4:\n            print(f\"skip {code}: only {len(rows)} train rows ({source})\", flush=True)\n            continue\n        path, losses = train_one_language(manager, code, rows, steps=steps)\n        if path is None:\n            print(f\"skip {code}: no usable examples\", flush=True)\n            continue\n        first = sum(losses[:5]) / len(losses[:5])\n        last = sum(losses[-5:]) / len(losses[-5:])\n        trained[code] = {\"n\": len(rows), \"source\": source,\n                         \"loss_first\": first, \"loss_last\": last}\n        print(f\"{code:3s} ({lang_name:10s}) n={len(rows):3d} \"\n              f\"loss {first:6.3f} -> {last:6.3f}\", flush=True)\n    return trained\n\n\nif __name__ == \"__main__\":\n    import argparse\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--steps\", type=int, default=60)\n    ap.add_argument(\"--n-train\", type=int, default=48)\n    ap.add_argument(\"--force\", action=\"store_true\",\n                    help=\"retrain languages that already have an adapter\")\n    a = ap.parse_args()\n    train_all(steps=a.steps, n_train=a.n_train, force=a.force)\n", "evaluate.py": "\"\"\"Shared evaluation routine used by both pipeline.py and optimize.py.\"\"\"\nimport collections\nimport sys\nimport time\n\nfrom router.config import DEFAULT\nfrom router.generation import build_prompt, generate_unified, extract_answer\n\n\ndef evaluate(manager, items, route=True, cfg=None, per_item=False,\n             label=\"\", progress_every=10):\n    \"\"\"Run the eval set.\n\n    Progress is printed periodically: a 1500-generation sweep otherwise looks\n    identical to a hang for hours, which is how the first Kaggle run was lost.\n    Set progress_every=0 to silence it.\n    \"\"\"\n    cfg = cfg or DEFAULT\n    correct = 0\n    tot_gen = tot_proc = tot_dispatch = tot_saved = tot_switch = 0\n    per_lang = collections.defaultdict(\n        lambda: {\"n\": 0, \"correct\": 0, \"gen\": 0, \"proc\": 0, \"switches\": 0}\n    )\n    items_out = []\n    started = time.time()\n\n    for idx, row in enumerate(items, 1):\n        if progress_every and (idx == 1 or idx % progress_every == 0):\n            elapsed = time.time() - started\n            rate = (idx - 1) / elapsed if elapsed > 0 and idx > 1 else 0\n            eta = (len(items) - idx) / rate if rate > 0 else float(\"nan\")\n            print(f\"  [{label or ('router' if route else 'baseline')}] \"\n                  f\"{idx}/{len(items)}  \"\n                  f\"acc={correct / max(idx - 1, 1):.3f}  \"\n                  f\"{rate * 60:.1f} items/min  eta {eta / 60:.0f}m\",\n                  flush=True, file=sys.stderr)\n        prompt = build_prompt(manager.tokenizer, row[\"question\"], row[\"options\"])\n        # Seed the initial expert from the question itself, not the templated\n        # prompt (which is dominated by English instruction scaffolding).\n        r = generate_unified(manager, prompt, cfg=cfg, route=route,\n                             seed_text=row[\"question\"])\n        pred = extract_answer(r[\"text\"])\n        hit = int(pred == row[\"answer\"])\n\n        correct += hit\n        tot_gen += r[\"num_tokens\"]\n        tot_proc += r[\"total_processed\"]\n        tot_dispatch += r[\"dispatch_total_processed\"]\n        tot_saved += r[\"prefill_saved\"]\n        tot_switch += len(r[\"switches\"])\n\n        st = per_lang[row[\"lang\"]]\n        st[\"n\"] += 1\n        st[\"correct\"] += hit\n        st[\"gen\"] += r[\"num_tokens\"]\n        st[\"proc\"] += r[\"total_processed\"]\n        st[\"switches\"] += len(r[\"switches\"])\n\n        if per_item:\n            items_out.append({\n                \"lang\": row[\"lang\"], \"gold\": row[\"answer\"], \"pred\": pred,\n                \"correct\": hit, \"switches\": r[\"switches\"],\n                \"text\": r[\"text\"][:400],\n            })\n\n    n = len(items) or 1\n    out = {\n        \"accuracy\": correct / n,\n        \"avg_gen_tokens\": tot_gen / n,\n        \"avg_total_processed\": tot_proc / n,\n        \"avg_dispatch_processed\": tot_dispatch / n,\n        \"avg_prefill_saved\": tot_saved / n,\n        \"token_saving_ratio\": (tot_dispatch - tot_proc) / tot_dispatch if tot_dispatch else 0.0,\n        \"avg_switches\": tot_switch / n,\n        \"n\": len(items),\n        \"unanswered\": sum(1 for i in items_out if i[\"pred\"] is None) if per_item else None,\n        \"per_lang\": {\n            k: {\n                \"acc\": v[\"correct\"] / v[\"n\"],\n                \"avg_gen\": v[\"gen\"] / v[\"n\"],\n                \"avg_proc\": v[\"proc\"] / v[\"n\"],\n                \"avg_switches\": v[\"switches\"] / v[\"n\"],\n                \"n\": v[\"n\"],\n            }\n            for k, v in sorted(per_lang.items())\n        },\n    }\n    if per_item:\n        out[\"items\"] = items_out\n    return out\n\n\ndef summarize(result):\n    \"\"\"Drop the bulky per-item payload for sweep logs.\"\"\"\n    return {k: v for k, v in result.items() if k != \"items\"}\n", "pipeline.py": "\"\"\"Evaluation entrypoint.\n\nCompares two configurations on the same items, same prompts, same decoding:\n\n  baseline  one fixed set of weights for the whole generation (no routing)\n  router    unified CoT with the active LoRA expert hot-swapped mid-generation\n            whenever the language-shift signal fires, over one continuous\n            token history and one continuous KV cache\n\nBenchmarks: MILU (11 Indic languages) + MMMLU (9 non-Indic), both 4-option MCQ.\n\nRun: python3 pipeline.py\nScores land in results/history.json; the run reports whether the last two runs\nhave plateaued on accuracy and token count.\n\"\"\"\nimport argparse\nimport json\nimport os\nimport sys\n\nfrom evaluate import evaluate, summarize\nfrom router.benchmarks import load_eval_set\nfrom router.config import RouterConfig, DEFAULT\nfrom router.model_manager import ExpertManager\n\nRESULTS_DIR = os.path.join(os.path.dirname(__file__), \"results\")\nHISTORY_PATH = os.path.join(RESULTS_DIR, \"history.json\")\nBEST_CONFIG_PATH = os.path.join(RESULTS_DIR, \"sweep.json\")\nACC_EPS = 0.01\nTOK_EPS = 1.0\nPLATEAU_RUNS = 2\n\n\ndef load_best_config():\n    \"\"\"Prefer the config the sweep settled on, if one exists.\"\"\"\n    if os.path.isfile(BEST_CONFIG_PATH):\n        try:\n            with open(BEST_CONFIG_PATH) as f:\n                return RouterConfig(**json.load(f)[\"best\"]), \"sweep\"\n        except Exception:\n            pass\n    return DEFAULT, \"default\"\n\n\ndef load_history():\n    if os.path.isfile(HISTORY_PATH):\n        with open(HISTORY_PATH) as f:\n            return json.load(f)\n    return []\n\n\ndef save_history(history):\n    os.makedirs(RESULTS_DIR, exist_ok=True)\n    with open(HISTORY_PATH, \"w\") as f:\n        json.dump(history, f, indent=2, ensure_ascii=False)\n\n\ndef check_plateau(history):\n    if len(history) < PLATEAU_RUNS:\n        return False\n    for a, b in zip(history[-PLATEAU_RUNS:], history[-PLATEAU_RUNS + 1:]):\n        if abs(a[\"router\"][\"accuracy\"] - b[\"router\"][\"accuracy\"]) > ACC_EPS:\n            return False\n        if abs(a[\"router\"][\"avg_total_processed\"] - b[\"router\"][\"avg_total_processed\"]) > TOK_EPS:\n            return False\n    return True\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--per-lang\", type=int, default=5)\n    ap.add_argument(\"--verbose\", action=\"store_true\")\n    args = ap.parse_args()\n\n    items, sources = load_eval_set(limit_per_lang=args.per_lang)\n    if not items:\n        print(\"No eval items available. Aborting.\")\n        return 1\n    langs = sorted({r[\"lang\"] for r in items})\n    print(f\"Eval set: {len(items)} items / {len(langs)} languages: {' '.join(langs)}\")\n\n    cfg, cfg_src = load_best_config()\n    print(f\"Router config ({cfg_src}): {cfg.to_dict()}\")\n\n    manager = ExpertManager()\n    print(f\"Device: {manager.device}\")\n    print(f\"Trained experts: {manager.available_experts() or '(none — base only)'}\\n\")\n\n    import dataclasses\n    recompute_cfg = dataclasses.replace(cfg, recompute_on_switch=True)\n\n    baseline = evaluate(manager, items, route=False, cfg=cfg,\n                        per_item=args.verbose, label=\"baseline\")\n    router = evaluate(manager, items, route=True, cfg=cfg,\n                      per_item=args.verbose, label=\"router\")\n    # Third arm: same routing decisions, but the cache is rebuilt under the new\n    # expert on each switch. Isolates the cost of reusing a history that a\n    # different expert encoded. Slowest arm by far — it re-prefills the whole\n    # prefix on every switch.\n    exact = evaluate(manager, items, route=True, cfg=recompute_cfg,\n                     per_item=args.verbose, label=\"recomputed\")\n\n    print(f\"Baseline (no routing)     : acc={baseline['accuracy']:.3f}  \"\n          f\"gen={baseline['avg_gen_tokens']:.1f}  processed={baseline['avg_total_processed']:.1f}\")\n    print(f\"Router   (shared cache)   : acc={router['accuracy']:.3f}  \"\n          f\"gen={router['avg_gen_tokens']:.1f}  processed={router['avg_total_processed']:.1f}  \"\n          f\"switches={router['avg_switches']:.2f}\")\n    print(f\"Router   (recomputed)     : acc={exact['accuracy']:.3f}  \"\n          f\"gen={exact['avg_gen_tokens']:.1f}  processed={exact['avg_total_processed']:.1f}  \"\n          f\"switches={exact['avg_switches']:.2f}\")\n\n    print(f\"\\nDecomposition:\")\n    print(f\"  value of experts   (recomputed - baseline): \"\n          f\"{exact['accuracy'] - baseline['accuracy']:+.3f} acc\")\n    print(f\"  cost of shared cache (router - recomputed): \"\n          f\"{router['accuracy'] - exact['accuracy']:+.3f} acc, \"\n          f\"{router['avg_total_processed'] - exact['avg_total_processed']:+.1f} tokens\")\n    print(f\"  net vs baseline      (router - baseline)  : \"\n          f\"{router['accuracy'] - baseline['accuracy']:+.3f} acc, \"\n          f\"{router['avg_total_processed'] - baseline['avg_total_processed']:+.1f} tokens\")\n\n    # The architectural claim: a dispatch design re-encodes the prefix on every\n    # switch, the unified design does not.\n    print(f\"\\nToken efficiency vs a re-encoding dispatch design:\")\n    print(f\"  unified processed/item : {router['avg_total_processed']:.1f}\")\n    print(f\"  dispatch processed/item: {router['avg_dispatch_processed']:.1f}\")\n    print(f\"  prefill saved/item     : {router['avg_prefill_saved']:.1f} \"\n          f\"({router['token_saving_ratio'] * 100:.1f}%)\")\n\n    print(\"\\nper-language (baseline acc -> router acc, switches):\")\n    for lang in langs:\n        b = baseline[\"per_lang\"].get(lang, {})\n        r = router[\"per_lang\"].get(lang, {})\n        print(f\"  {lang:3s} n={b.get('n', 0):3d}  {b.get('acc', 0):.2f} -> {r.get('acc', 0):.2f}\"\n              f\"   sw={r.get('avg_switches', 0):.2f}\")\n\n    history = load_history()\n    history.append({\n        \"config\": cfg.to_dict(),\n        \"config_source\": cfg_src,\n        \"baseline\": summarize(baseline),\n        \"router\": summarize(router),\n        \"router_recomputed\": summarize(exact),\n        \"sources\": sources,\n        \"experts\": manager.available_experts(),\n        \"per_lang_items\": args.per_lang,\n    })\n    save_history(history)\n\n    plateaued = check_plateau(history)\n    print(f\"\\nRun {len(history)} recorded. Plateau reached: {plateaued}\")\n    return 0\n\n\nif __name__ == \"__main__\":\n    sys.exit(main())\n", "optimize.py": "\"\"\"Outer optimization loop over router mechanics.\n\nRuns the eval once per candidate RouterConfig, keeps the best, and stops when\nthe best score stops improving for `--patience` consecutive rounds — i.e. the\nbenchmark score has plateaued.\n\nThe model is loaded once and reused across every candidate, so a sweep costs\none model load plus N evals.\n\nRun: python3 optimize.py --per-lang 5\n\"\"\"\nimport argparse\nimport itertools\nimport json\nimport os\nimport sys\n\nfrom router.benchmarks import load_eval_set\nfrom router.config import RouterConfig\nfrom router.model_manager import ExpertManager\nfrom evaluate import evaluate, summarize\n\nRESULTS_DIR = os.path.join(os.path.dirname(__file__), \"results\")\nSWEEP_PATH = os.path.join(RESULTS_DIR, \"sweep.json\")\n\n# Candidate values per knob. Kept small on purpose: each combination is a full\n# eval pass, so the grid is the expensive part.\nGRID = {\n    \"check_every\": [2, 4, 8],\n    \"window_chars\": [64, 128, 32],\n    \"switch_patience\": [1, 2],\n    \"latin_min_conf\": [0.90, 0.99],\n}\n\n\ndef is_viable(cfg):\n    \"\"\"Reject configurations that cannot express the behaviour being swept.\n\n    `window_chars < latin_min_chars` silently disables Latin detection\n    entirely: the detector never sees enough Latin characters to return a\n    verdict. Since the model reasons in English, nearly every switch is *into*\n    `en`, so such a config collapses to ~0 switches and scores identically to\n    baseline. A first sweep wasted its whole budget on this corner and then\n    declared a \"plateau\" at the resulting flat score.\n    \"\"\"\n    if cfg.window_chars < cfg.latin_min_chars:\n        return False, (f\"window_chars={cfg.window_chars} < \"\n                       f\"latin_min_chars={cfg.latin_min_chars}: \"\n                       f\"Latin script undetectable, routing disabled\")\n    return True, None\n\n\ndef candidates():\n    \"\"\"Viable grid points. Dropped combinations are returned too, so the\n    caller can report them rather than silently narrowing the search.\"\"\"\n    keys = list(GRID)\n    good, dropped = [], []\n    for combo in itertools.product(*(GRID[k] for k in keys)):\n        cfg = RouterConfig(**dict(zip(keys, combo)))\n        ok, why = is_viable(cfg)\n        (good if ok else dropped).append(cfg if ok else (cfg, why))\n    return good, dropped\n\n\ndef score_of(result):\n    \"\"\"Primary objective is accuracy; ties break toward fewer tokens\n    processed, since token efficiency is the secondary goal.\"\"\"\n    return (result[\"accuracy\"], -result[\"avg_total_processed\"])\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--per-lang\", type=int, default=3)\n    ap.add_argument(\"--patience\", type=int, default=3,\n                    help=\"rounds without improvement before declaring plateau\")\n    ap.add_argument(\"--max-candidates\", type=int, default=None)\n    args = ap.parse_args()\n\n    items, sources = load_eval_set(limit_per_lang=args.per_lang)\n    if not items:\n        print(\"No eval items available. Aborting.\")\n        return 1\n    langs = sorted({r[\"lang\"] for r in items})\n    print(f\"Eval set: {len(items)} items / {len(langs)} languages\")\n\n    manager = ExpertManager()\n    print(f\"Experts: {manager.available_experts() or '(none — base only)'}\")\n\n    cands, dropped = candidates()\n    for cfg, why in dropped:\n        print(f\"  dropped {cfg.check_every}/{cfg.window_chars}/\"\n              f\"{cfg.switch_patience}/{cfg.latin_min_conf}: {why}\")\n    if dropped:\n        print(f\"  ({len(dropped)} unviable configurations excluded)\\n\")\n    if args.max_candidates:\n        cands = cands[: args.max_candidates]\n    print(f\"Sweeping {len(cands)} router configurations\\n\")\n\n    history = []\n    best, best_score, stale = None, None, 0\n    distinct_scores = set()\n\n    for i, cfg in enumerate(cands, 1):\n        result = evaluate(manager, items, route=True, cfg=cfg,\n                          label=f\"cfg{i}\")\n        s = score_of(result)\n        improved = best_score is None or s > best_score\n\n        if improved:\n            best, best_score, stale = cfg, s, 0\n        else:\n            # Identical scores mean the knob under test did nothing, which is\n            # not evidence of a plateau — the earlier sweep \"converged\" purely\n            # because a degenerate corner of the grid produced four identical\n            # numbers in a row. Only count a genuinely new-but-worse score\n            # against the patience budget.\n            if round(s[0], 4) in distinct_scores:\n                print(f\"    (score unchanged — not counted toward plateau)\")\n            else:\n                stale += 1\n        distinct_scores.add(round(s[0], 4))\n\n        history.append({\"config\": cfg.to_dict(), \"result\": summarize(result),\n                        \"improved\": improved})\n        print(f\"[{i:>2}/{len(cands)}] acc={result['accuracy']:.3f} \"\n              f\"tok={result['avg_total_processed']:.0f} \"\n              f\"sw={result['avg_switches']:.2f} \"\n              f\"{'*BEST*' if improved else f'stale {stale}/{args.patience}'}\"\n              f\"  {cfg.to_dict()}\")\n\n        if stale >= args.patience:\n            print(f\"\\nPlateau: no improvement in {args.patience} consecutive rounds.\")\n            break\n\n    os.makedirs(RESULTS_DIR, exist_ok=True)\n    with open(SWEEP_PATH, \"w\") as f:\n        json.dump({\"sources\": sources, \"best\": best.to_dict(),\n                   \"best_score\": {\"accuracy\": best_score[0],\n                                  \"avg_total_processed\": -best_score[1]},\n                   \"history\": history}, f, indent=2, ensure_ascii=False)\n\n    print(f\"\\nBest config: {best.to_dict()}\")\n    print(f\"Best accuracy: {best_score[0]:.3f}  avg_total_processed: {-best_score[1]:.0f}\")\n    print(f\"Sweep written to {SWEEP_PATH}\")\n    return 0\n\n\nif __name__ == \"__main__\":\n    sys.exit(main())\n", "tests/test_router.py": "\"\"\"Tests that need no base model — detector behaviour and data normalization.\"\"\"\nimport os\nimport sys\n\nsys.path.insert(0, os.path.join(os.path.dirname(__file__), \"..\"))\n\nfrom router.config import RouterConfig\nfrom router.script_detect import detect_language, language_shifted\n\nCASES = [\n    (\"中国的首都是北京，人口很多\", \"zh\"),\n    (\"日本の首都は東京です\", \"ja\"),\n    # Majority-Han Japanese: the Kana rule must beat the dominant-script rule.\n    (\"東京は日本の首都で、人口が多い\", \"ja\"),\n    (\"대한민국의 수도는 서울입니다\", \"ko\"),\n    (\"العاصمة هي الرياض والمدينة كبيرة\", \"ar\"),\n    (\"भारत की राजधानी दिल्ली है\", \"hi\"),\n    (\"मी मराठी बोलतो आणि लिहितो चांगले आहे\", \"mr\"),\n    (\"হে একটি ভাল উদাহরণ এবং আমি জানি না\", \"bn\"),\n    (\"இந்தியாவின் தலைநகரம் எது\", \"ta\"),\n    (\"ಕರ್ನಾಟಕದ ರಾಜಧಾನಿ ಯಾವುದು\", \"kn\"),\n    (\"The capital of India is New Delhi and it is a very large city\", \"en\"),\n    (\"Die Hauptstadt von Deutschland ist Berlin und sie ist sehr schoen\", \"de\"),\n    (\"La capitale de la France est Paris et elle est tres belle ville\", \"fr\"),\n]\n\nUNDECIDED = [\"\", \"   \", \"42\", \"3.14159\", \"A. B. C.\", \"short\"]\n\n\ndef test_detection():\n    fails = []\n    for text, expected in CASES:\n        got = detect_language(text)\n        if got != expected:\n            fails.append(f\"  {text[:40]!r} -> {got} (expected {expected})\")\n    assert not fails, \"detection failures:\\n\" + \"\\n\".join(fails)\n\n\ndef test_undecided_returns_none():\n    \"\"\"Too little evidence must yield None so the router holds the current\n    expert rather than defaulting to one.\"\"\"\n    for text in UNDECIDED:\n        assert detect_language(text) is None, f\"{text!r} should be undecided\"\n\n\ndef test_latin_needs_more_evidence_than_unique_script():\n    \"\"\"A short Latin fragment is undecidable, but the same length of a\n    script-unique language is decidable.\"\"\"\n    assert detect_language(\"The capital\") is None\n    assert detect_language(\"中国的首都是北京\") == \"zh\"\n\n\ndef test_no_shift_when_language_unchanged():\n    text = \"The capital of India is New Delhi and it is a very large city\"\n    assert language_shifted(\"en\", text) is None\n\n\ndef test_shift_detected_on_change():\n    text = \"The capital of India is New Delhi and it is a very large city\"\n    assert language_shifted(\"hi\", text) == \"en\"\n\n\ndef test_config_threshold_is_respected():\n    \"\"\"Raising the Latin confidence floor must make borderline text undecided.\"\"\"\n    strict = RouterConfig(latin_min_conf=1.01)  # unreachable by construction\n    text = \"The capital of India is New Delhi and it is a very large city\"\n    assert language_shifted(\"hi\", text, strict) is None\n\n\ndef test_answer_extraction():\n    from router.generation import extract_answer\n    assert extract_answer(\"blah\\nAnswer: C\") == \"C\"\n    assert extract_answer(\"Answer: (B)\") == \"B\"\n    assert extract_answer(\"answer - d\") == \"D\"\n    assert extract_answer(\"no answer here\") is None\n\n\ndef test_benchmark_normalization():\n    from router.benchmarks import _norm_milu, _norm_mmmlu\n    milu = _norm_milu(\n        {\"question\": \"q\", \"option1\": \"a\", \"option2\": \"b\", \"option3\": \"c\",\n         \"option4\": \"d\", \"target\": \"option3\", \"subject\": \"s\"},\n        \"Hindi\", \"hi\")\n    assert milu[\"answer\"] == \"C\" and milu[\"options\"][2] == \"c\"\n\n    mmmlu = _norm_mmmlu(\n        {\"Question\": \"q\", \"A\": \"a\", \"B\": \"b\", \"C\": \"c\", \"D\": \"d\",\n         \"Answer\": \"B\", \"Subject\": \"s\"},\n        \"DE_DE\", \"de\")\n    assert mmmlu[\"answer\"] == \"B\" and mmmlu[\"options\"][1] == \"b\"\n\n\ndef test_sweep_excludes_configs_that_disable_routing():\n    \"\"\"A window shorter than the Latin evidence floor makes Latin\n    undetectable. Since the model reasons in English, that silently collapses\n    routing to ~0 switches and the whole config scores as baseline — which a\n    first sweep then mistook for a plateau.\"\"\"\n    from optimize import candidates, is_viable\n    from router.config import RouterConfig\n\n    ok, why = is_viable(RouterConfig(window_chars=32, latin_min_chars=40))\n    assert not ok and \"undetectable\" in why\n\n    assert is_viable(RouterConfig(window_chars=64, latin_min_chars=40))[0]\n\n    good, dropped = candidates()\n    assert dropped, \"grid should exclude at least one unviable config\"\n    for cfg in good:\n        assert cfg.window_chars >= cfg.latin_min_chars\n\n\ndef test_latin_undetectable_below_floor():\n    \"\"\"The underlying reason the above configs are dead: a Latin window\n    shorter than the floor yields no verdict at all.\"\"\"\n    text = \"The capital of India is New Delhi\"      # < 40 Latin chars\n    assert detect_language(text, latin_min_chars=40) is None\n    assert detect_language(text, latin_min_chars=10) == \"en\"\n\n\ndef test_milu_numeric_target_form():\n    \"\"\"The fixture uses '2'; the real dataset uses 'option2'. Both must work.\"\"\"\n    from router.benchmarks import _norm_milu\n    row = {\"question\": \"q\", \"option1\": \"a\", \"option2\": \"b\", \"option3\": \"c\",\n           \"option4\": \"d\", \"target\": \"2\", \"subject\": \"s\"}\n    assert _norm_milu(row, \"Hindi\", \"hi\")[\"answer\"] == \"B\"\n", "tests/test_switching.py": "\"\"\"Mechanism tests for the weight-switching architecture.\n\nThese run on a tiny model, so they verify the *mechanism* — cache continuity\nacross a swap, real weight deltas, grad isolation — without needing the real\nbase model. This is the part of the design most likely to be silently wrong.\n\"\"\"\nimport os\nimport sys\n\nimport pytest\nimport torch\n\nsys.path.insert(0, os.path.join(os.path.dirname(__file__), \"..\"))\n\nfrom peft import LoraConfig, get_peft_model\nfrom transformers import AutoModelForCausalLM, AutoTokenizer\n\nTINY = \"sshleifer/tiny-gpt2\"\n\n\n@pytest.fixture(scope=\"module\")\ndef tiny():\n    tok = AutoTokenizer.from_pretrained(TINY)\n    if tok.pad_token is None:\n        tok.pad_token = tok.eos_token\n    base = AutoModelForCausalLM.from_pretrained(TINY, dtype=torch.float32)\n    cfg = LoraConfig(r=4, lora_alpha=8, lora_dropout=0.0,\n                     target_modules=[\"c_attn\"], task_type=\"CAUSAL_LM\")\n    model = get_peft_model(base, cfg, adapter_name=\"_base\")\n    for name in (\"hi\", \"zh\"):\n        model.add_adapter(name, model.peft_config[\"_base\"])\n    model.eval()\n    return tok, model\n\n\ndef test_base_adapter_is_identity(tiny):\n    \"\"\"The baseline arm must be the genuine unmodified model, so the seed\n    adapter's B matrices have to be zero (making its delta exactly zero).\"\"\"\n    _, model = tiny\n    zeros = [p for n, p in model.named_parameters()\n             if \"lora_B\" in n and \"_base\" in n]\n    assert zeros, \"no _base lora_B parameters found\"\n    assert all(float(p.abs().sum()) == 0.0 for p in zeros)\n\n\ndef test_grad_isolation(tiny):\n    \"\"\"Training one expert must not drag gradients into the others.\"\"\"\n    _, model = tiny\n    for active in (\"hi\", \"zh\"):\n        model.set_adapter(active)\n        trainable = {n.split(\".\")[-2] for n, p in model.named_parameters()\n                     if p.requires_grad}\n        assert trainable == {active}, f\"active={active} but trainable={trainable}\"\n\n\ndef test_cache_survives_adapter_swap(tiny):\n    \"\"\"The load-bearing property: keys/values computed under one expert must\n    remain valid input after swapping to another, so the token history is\n    never re-encoded.\"\"\"\n    tok, model = tiny\n    ids = tok(\"the capital of india is\", return_tensors=\"pt\").input_ids\n\n    model.set_adapter(\"hi\")\n    with torch.no_grad():\n        out = model(input_ids=ids, use_cache=True)\n        past = out.past_key_values\n        nxt = int(out.logits[0, -1].argmax())\n\n        model.set_adapter(\"zh\")\n        cont = model(input_ids=torch.tensor([[nxt]]),\n                     past_key_values=past, use_cache=True)\n\n    assert cont.logits.shape[:2] == (1, 1)\n    assert torch.isfinite(cont.logits).all()\n\n\ndef test_swapping_changes_output_once_experts_differ(tiny):\n    \"\"\"A swap has to actually change the computation — otherwise the router is\n    measuring nothing. Perturb one adapter and confirm logits move.\"\"\"\n    tok, model = tiny\n    ids = tok(\"hello world\", return_tensors=\"pt\").input_ids\n\n    model.set_adapter(\"_base\")\n    with torch.no_grad():\n        base_logits = model(input_ids=ids).logits.clone()\n\n    # Give \"hi\" a non-zero delta, the way training would.\n    with torch.no_grad():\n        for n, p in model.named_parameters():\n            if \"lora_B\" in n and \"hi\" in n:\n                p.add_(torch.randn_like(p) * 0.5)\n\n    model.set_adapter(\"hi\")\n    with torch.no_grad():\n        hi_logits = model(input_ids=ids).logits\n\n    assert not torch.allclose(base_logits, hi_logits), \\\n        \"adapter swap did not change the output\"\n\n    # And switching back must restore the base behaviour exactly.\n    model.set_adapter(\"_base\")\n    with torch.no_grad():\n        back = model(input_ids=ids).logits\n    assert torch.allclose(base_logits, back, atol=1e-6)\n\n\ndef test_cache_reuse_matches_full_forward(tiny):\n    \"\"\"Incremental decoding with a retained cache must agree with a single\n    full-sequence forward pass, when no swap intervenes. This guards the\n    cache plumbing itself.\"\"\"\n    tok, model = tiny\n    model.set_adapter(\"_base\")\n    ids = tok(\"the capital of india is new delhi\", return_tensors=\"pt\").input_ids\n\n    with torch.no_grad():\n        full = model(input_ids=ids).logits[:, -1, :]\n\n        step = model(input_ids=ids[:, :-1], use_cache=True)\n        inc = model(input_ids=ids[:, -1:],\n                    past_key_values=step.past_key_values).logits[:, -1, :]\n\n    assert torch.allclose(full, inc, atol=1e-4), \\\n        \"incremental cache path diverges from full forward\"\n", "tests/test_pipeline_smoke.py": "\"\"\"End-to-end smoke test on a tiny model.\n\nExercises the real generation loop, the real evaluate() and the real config\nplumbing, so integration bugs surface without waiting on the full base model.\nAccuracy is meaningless here — only that every arm runs and the accounting\nis self-consistent.\n\"\"\"\nimport os\nimport sys\n\nimport pytest\nimport torch\n\nsys.path.insert(0, os.path.join(os.path.dirname(__file__), \"..\"))\n\nfrom peft import LoraConfig, get_peft_model\nfrom transformers import AutoModelForCausalLM, AutoTokenizer\n\nfrom evaluate import evaluate\nfrom router.config import RouterConfig\n\nTINY = \"sshleifer/tiny-gpt2\"\n\n\nclass TinyManager:\n    \"\"\"Same surface as ExpertManager, backed by a tiny model.\"\"\"\n\n    def __init__(self):\n        self.device = \"cpu\"\n        self.tokenizer = AutoTokenizer.from_pretrained(TINY)\n        if self.tokenizer.pad_token is None:\n            self.tokenizer.pad_token = self.tokenizer.eos_token\n        # tiny-gpt2 has no chat template; supply a minimal one.\n        self.tokenizer.chat_template = (\n            \"{% for m in messages %}{{ m['content'] }}\\n{% endfor %}\"\n        )\n        base = AutoModelForCausalLM.from_pretrained(TINY, dtype=torch.float32)\n        cfg = LoraConfig(r=4, lora_alpha=8, lora_dropout=0.0,\n                         target_modules=[\"c_attn\"], task_type=\"CAUSAL_LM\")\n        self.model = get_peft_model(base, cfg, adapter_name=\"_base\")\n        self.loaded = {\"_base\"}\n        for name in (\"hi\", \"zh\", \"en\"):\n            self.model.add_adapter(name, self.model.peft_config[\"_base\"])\n            self.loaded.add(name)\n        self.model.eval()\n        self.active = \"_base\"\n\n    def available_experts(self):\n        return sorted(self.loaded - {\"_base\"})\n\n    def set_active(self, lang):\n        target = lang if lang in self.loaded else \"_base\"\n        if target != self.active:\n            self.model.set_adapter(target)\n            self.active = target\n        return self.active\n\n\nITEMS = [\n    {\"question\": \"भारत की राजधानी क्या है?\",\n     \"options\": [\"मुंबई\", \"नई दिल्ली\", \"कोलकाता\", \"चेन्नई\"],\n     \"answer\": \"B\", \"lang\": \"hi\", \"lang_name\": \"Hindi\", \"subject\": \"geo\",\n     \"source\": \"test\"},\n    {\"question\": \"中国的首都是什么？\",\n     \"options\": [\"上海\", \"北京\", \"广州\", \"深圳\"],\n     \"answer\": \"B\", \"lang\": \"zh\", \"lang_name\": \"ZH_CN\", \"subject\": \"geo\",\n     \"source\": \"test\"},\n    {\"question\": \"What is the capital of India?\",\n     \"options\": [\"Mumbai\", \"New Delhi\", \"Kolkata\", \"Chennai\"],\n     \"answer\": \"B\", \"lang\": \"en\", \"lang_name\": \"English\", \"subject\": \"geo\",\n     \"source\": \"test\"},\n]\n\n\n@pytest.fixture(scope=\"module\")\ndef manager():\n    return TinyManager()\n\n\n@pytest.fixture(scope=\"module\")\ndef cfg():\n    return RouterConfig(max_new_tokens=12, check_every=2)\n\n\ndef test_baseline_arm_runs(manager, cfg):\n    r = evaluate(manager, ITEMS, route=False, cfg=cfg)\n    assert r[\"n\"] == len(ITEMS)\n    assert 0.0 <= r[\"accuracy\"] <= 1.0\n    # No routing means no switches and therefore no notional prefill saving.\n    assert r[\"avg_switches\"] == 0\n    assert r[\"avg_prefill_saved\"] == 0\n\n\ndef test_router_arm_runs(manager, cfg):\n    r = evaluate(manager, ITEMS, route=True, cfg=cfg)\n    assert r[\"n\"] == len(ITEMS)\n    assert set(r[\"per_lang\"]) == {\"hi\", \"zh\", \"en\"}\n\n\ndef test_recompute_arm_runs(manager, cfg):\n    import dataclasses\n    r = evaluate(manager, ITEMS, route=True,\n                 cfg=dataclasses.replace(cfg, recompute_on_switch=True))\n    assert r[\"n\"] == len(ITEMS)\n\n\ndef test_token_accounting_is_consistent(manager, cfg):\n    \"\"\"Dispatch must never be cheaper than unified, and the saving must equal\n    the stated difference.\"\"\"\n    r = evaluate(manager, ITEMS, route=True, cfg=cfg)\n    assert r[\"avg_dispatch_processed\"] >= r[\"avg_total_processed\"] - 1e-9\n    assert r[\"avg_prefill_saved\"] >= 0\n    assert 0.0 <= r[\"token_saving_ratio\"] < 1.0\n\n\ndef test_baseline_and_router_see_identical_items(manager, cfg):\n    a = evaluate(manager, ITEMS, route=False, cfg=cfg, per_item=True)\n    b = evaluate(manager, ITEMS, route=True, cfg=cfg, per_item=True)\n    assert [i[\"gold\"] for i in a[\"items\"]] == [i[\"gold\"] for i in b[\"items\"]]\n    assert [i[\"lang\"] for i in a[\"items\"]] == [i[\"lang\"] for i in b[\"items\"]]\n\n\ndef test_max_new_tokens_respected(manager):\n    r = evaluate(manager, ITEMS, route=True, cfg=RouterConfig(max_new_tokens=5))\n    assert r[\"avg_gen_tokens\"] <= 5\n\n\ndef test_routing_does_not_collapse_generation(manager, cfg):\n    \"\"\"Guard against adapters that destroy instruction-following.\n\n    Over-trained experts (lr=1e-4 for 60 steps) made routed runs emit a bare\n    \"Answer: C\" in 3 tokens while the baseline still produced full reasoning.\n    That looks like a huge token saving in the metrics but is just a broken\n    model, so a large collapse in generated length is treated as a failure\n    rather than a win.\n    \"\"\"\n    base = evaluate(manager, ITEMS, route=False, cfg=cfg)\n    routed = evaluate(manager, ITEMS, route=True, cfg=cfg)\n    assert routed[\"avg_gen_tokens\"] >= 0.4 * base[\"avg_gen_tokens\"], (\n        f\"routed generation collapsed: {routed['avg_gen_tokens']:.1f} vs \"\n        f\"baseline {base['avg_gen_tokens']:.1f} tokens\"\n    )\n\n\ndef test_seed_text_selects_initial_expert(manager, cfg):\n    \"\"\"The initial expert must come from the question, not the templated\n    prompt — the latter is mostly English scaffolding and misroutes.\"\"\"\n    from router.generation import build_prompt, generate_unified\n    q = \"中国的首都是什么？\"\n    prompt = build_prompt(manager.tokenizer, q, [\"上海\", \"北京\", \"广州\", \"深圳\"])\n    r = generate_unified(manager, prompt, cfg=cfg, route=True, seed_text=q)\n    assert r[\"switches\"] == [] or r[\"switches\"][0][\"from\"] == \"zh\"\n", "data/milu_stub.jsonl": "{\"question\": \"What is the capital of India?\", \"option1\": \"Mumbai\", \"option2\": \"New Delhi\", \"option3\": \"Kolkata\", \"option4\": \"Chennai\", \"target\": \"2\", \"subject\": \"geography\", \"language\": \"English\", \"split\": \"validation\"}\n{\"question\": \"Which planet is known as the Red Planet?\", \"option1\": \"Venus\", \"option2\": \"Jupiter\", \"option3\": \"Mars\", \"option4\": \"Saturn\", \"target\": \"3\", \"subject\": \"science\", \"language\": \"English\", \"split\": \"validation\"}\n{\"question\": \"How many states does India currently have?\", \"option1\": \"28\", \"option2\": \"29\", \"option3\": \"30\", \"option4\": \"26\", \"target\": \"1\", \"subject\": \"civics\", \"language\": \"English\", \"split\": \"validation\"}\n{\"question\": \"Who wrote the Indian national anthem?\", \"option1\": \"Bankim Chandra\", \"option2\": \"Rabindranath Tagore\", \"option3\": \"Sarojini Naidu\", \"option4\": \"Subhas Chandra Bose\", \"target\": \"2\", \"subject\": \"history\", \"language\": \"English\", \"split\": \"validation\"}\n{\"question\": \"What is the freezing point of water in Celsius?\", \"option1\": \"0\", \"option2\": \"32\", \"option3\": \"100\", \"option4\": \"-1\", \"target\": \"1\", \"subject\": \"science\", \"language\": \"English\", \"split\": \"validation\"}\n{\"question\": \"भारत की राजधानी क्या है?\", \"option1\": \"मुंबई\", \"option2\": \"नई दिल्ली\", \"option3\": \"कोलकाता\", \"option4\": \"चेन्नई\", \"target\": \"2\", \"subject\": \"geography\", \"language\": \"Hindi\", \"split\": \"validation\"}\n{\"question\": \"लाल ग्रह किसे कहा जाता है?\", \"option1\": \"शुक्र\", \"option2\": \"बृहस्पति\", \"option3\": \"मंगल\", \"option4\": \"शनि\", \"target\": \"3\", \"subject\": \"science\", \"language\": \"Hindi\", \"split\": \"validation\"}\n{\"question\": \"भारत का राष्ट्रीय गान किसने लिखा?\", \"option1\": \"बंकिम चंद्र\", \"option2\": \"रवींद्रनाथ टैगोर\", \"option3\": \"सरोजिनी नायडू\", \"option4\": \"सुभाष चंद्र बोस\", \"target\": \"2\", \"subject\": \"history\", \"language\": \"Hindi\", \"split\": \"validation\"}\n{\"question\": \"पानी का हिमांक बिंदु सेल्सियस में क्या है?\", \"option1\": \"0\", \"option2\": \"32\", \"option3\": \"100\", \"option4\": \"-1\", \"target\": \"1\", \"subject\": \"science\", \"language\": \"Hindi\", \"split\": \"validation\"}\n{\"question\": \"भारत में वर्तमान में कितने राज्य हैं?\", \"option1\": \"28\", \"option2\": \"29\", \"option3\": \"30\", \"option4\": \"26\", \"target\": \"1\", \"subject\": \"civics\", \"language\": \"Hindi\", \"split\": \"validation\"}\n{\"question\": \"ভারতের রাজধানী কী?\", \"option1\": \"মুম্বাই\", \"option2\": \"নয়াদিল্লি\", \"option3\": \"কলকাতা\", \"option4\": \"চেন্নাই\", \"target\": \"2\", \"subject\": \"geography\", \"language\": \"Bengali\", \"split\": \"validation\"}\n{\"question\": \"লাল গ্রহ কাকে বলা হয়?\", \"option1\": \"শুক্র\", \"option2\": \"বৃহস্পতি\", \"option3\": \"মঙ্গল\", \"option4\": \"শনি\", \"target\": \"3\", \"subject\": \"science\", \"language\": \"Bengali\", \"split\": \"validation\"}\n{\"question\": \"ভারতের জাতীয় সংগীত কে লিখেছিলেন?\", \"option1\": \"বঙ্কিমচন্দ্র\", \"option2\": \"রবীন্দ্রনাথ ঠাকুর\", \"option3\": \"সরোজিনী নাইডু\", \"option4\": \"সুভাষচন্দ্র বসু\", \"target\": \"2\", \"subject\": \"history\", \"language\": \"Bengali\", \"split\": \"validation\"}\n{\"question\": \"জলের হিমাঙ্ক বিন্দু সেলসিয়াসে কত?\", \"option1\": \"0\", \"option2\": \"32\", \"option3\": \"100\", \"option4\": \"-1\", \"target\": \"1\", \"subject\": \"science\", \"language\": \"Bengali\", \"split\": \"validation\"}\n{\"question\": \"ভারতে বর্তমানে কতগুলি রাজ্য আছে?\", \"option1\": \"28\", \"option2\": \"29\", \"option3\": \"30\", \"option4\": \"26\", \"target\": \"1\", \"subject\": \"civics\", \"language\": \"Bengali\", \"split\": \"validation\"}\n{\"question\": \"இந்தியாவின் தலைநகரம் எது?\", \"option1\": \"மும்பை\", \"option2\": \"புது தில்லி\", \"option3\": \"கொல்கத்தா\", \"option4\": \"சென்னை\", \"target\": \"2\", \"subject\": \"geography\", \"language\": \"Tamil\", \"split\": \"validation\"}\n{\"question\": \"செவ்வாய் கிரகம் என்று அழைக்கப்படுவது எது?\", \"option1\": \"வெள்ளி\", \"option2\": \"வியாழன்\", \"option3\": \"செவ்வாய்\", \"option4\": \"சனி\", \"target\": \"3\", \"subject\": \"science\", \"language\": \"Tamil\", \"split\": \"validation\"}\n{\"question\": \"இந்திய தேசிய கீதத்தை எழுதியவர் யார்?\", \"option1\": \"பங்கிம் சந்திரா\", \"option2\": \"ரவீந்திரநாத் தாகூர்\", \"option3\": \"சரோஜினி நாயுடு\", \"option4\": \"சுபாஷ் சந்திர போஸ்\", \"target\": \"2\", \"subject\": \"history\", \"language\": \"Tamil\", \"split\": \"validation\"}\n{\"question\": \"நீரின் உறைநிலை செல்சியஸில் என்ன?\", \"option1\": \"0\", \"option2\": \"32\", \"option3\": \"100\", \"option4\": \"-1\", \"target\": \"1\", \"subject\": \"science\", \"language\": \"Tamil\", \"split\": \"validation\"}\n{\"question\": \"இந்தியாவில் தற்போது எத்தனை மாநிலங்கள் உள்ளன?\", \"option1\": \"28\", \"option2\": \"29\", \"option3\": \"30\", \"option4\": \"26\", \"target\": \"1\", \"subject\": \"civics\", \"language\": \"Tamil\", \"split\": \"validation\"}\n{\"question\": \"What is the largest ocean on Earth?\", \"option1\": \"Atlantic\", \"option2\": \"Indian\", \"option3\": \"Arctic\", \"option4\": \"Pacific\", \"target\": \"4\", \"subject\": \"geography\", \"language\": \"English\", \"split\": \"train\"}\n{\"question\": \"How many continents are there on Earth?\", \"option1\": \"5\", \"option2\": \"6\", \"option3\": \"7\", \"option4\": \"8\", \"target\": \"3\", \"subject\": \"geography\", \"language\": \"English\", \"split\": \"train\"}\n{\"question\": \"What gas do plants absorb from the air?\", \"option1\": \"Oxygen\", \"option2\": \"Nitrogen\", \"option3\": \"Carbon dioxide\", \"option4\": \"Hydrogen\", \"target\": \"3\", \"subject\": \"science\", \"language\": \"English\", \"split\": \"train\"}\n{\"question\": \"Who was the first Prime Minister of India?\", \"option1\": \"Mahatma Gandhi\", \"option2\": \"Jawaharlal Nehru\", \"option3\": \"Sardar Patel\", \"option4\": \"Indira Gandhi\", \"target\": \"2\", \"subject\": \"history\", \"language\": \"English\", \"split\": \"train\"}\n{\"question\": \"How many days are there in a leap year?\", \"option1\": \"365\", \"option2\": \"364\", \"option3\": \"366\", \"option4\": \"360\", \"target\": \"3\", \"subject\": \"science\", \"language\": \"English\", \"split\": \"train\"}\n{\"question\": \"पृथ्वी पर सबसे बड़ा महासागर कौन सा है?\", \"option1\": \"अटलांटिक\", \"option2\": \"हिंद महासागर\", \"option3\": \"आर्कटिक\", \"option4\": \"प्रशांत महासागर\", \"target\": \"4\", \"subject\": \"geography\", \"language\": \"Hindi\", \"split\": \"train\"}\n{\"question\": \"पृथ्वी पर कितने महाद्वीप हैं?\", \"option1\": \"5\", \"option2\": \"6\", \"option3\": \"7\", \"option4\": \"8\", \"target\": \"3\", \"subject\": \"geography\", \"language\": \"Hindi\", \"split\": \"train\"}\n{\"question\": \"पौधे हवा से कौन सी गैस अवशोषित करते हैं?\", \"option1\": \"ऑक्सीजन\", \"option2\": \"नाइट्रोजन\", \"option3\": \"कार्बन डाइऑक्साइड\", \"option4\": \"हाइड्रोजन\", \"target\": \"3\", \"subject\": \"science\", \"language\": \"Hindi\", \"split\": \"train\"}\n{\"question\": \"भारत के पहले प्रधानमंत्री कौन थे?\", \"option1\": \"महात्मा गांधी\", \"option2\": \"जवाहरलाल नेहरू\", \"option3\": \"सरदार पटेल\", \"option4\": \"इंदिरा गांधी\", \"target\": \"2\", \"subject\": \"history\", \"language\": \"Hindi\", \"split\": \"train\"}\n{\"question\": \"एक लीप वर्ष में कितने दिन होते हैं?\", \"option1\": \"365\", \"option2\": \"364\", \"option3\": \"366\", \"option4\": \"360\", \"target\": \"3\", \"subject\": \"science\", \"language\": \"Hindi\", \"split\": \"train\"}\n{\"question\": \"পৃথিবীর বৃহত্তম মহাসাগর কোনটি?\", \"option1\": \"আটলান্টিক\", \"option2\": \"ভারত মহাসাগর\", \"option3\": \"আর্কটিক\", \"option4\": \"প্রশান্ত মহাসাগর\", \"target\": \"4\", \"subject\": \"geography\", \"language\": \"Bengali\", \"split\": \"train\"}\n{\"question\": \"পৃথিবীতে কয়টি মহাদেশ আছে?\", \"option1\": \"5\", \"option2\": \"6\", \"option3\": \"7\", \"option4\": \"8\", \"target\": \"3\", \"subject\": \"geography\", \"language\": \"Bengali\", \"split\": \"train\"}\n{\"question\": \"গাছপালা বাতাস থেকে কোন গ্যাস শোষণ করে?\", \"option1\": \"অক্সিজেন\", \"option2\": \"নাইট্রোজেন\", \"option3\": \"কার্বন ডাইঅক্সাইড\", \"option4\": \"হাইড্রোজেন\", \"target\": \"3\", \"subject\": \"science\", \"language\": \"Bengali\", \"split\": \"train\"}\n{\"question\": \"ভারতের প্রথম প্রধানমন্ত্রী কে ছিলেন?\", \"option1\": \"মহাত্মা গান্ধী\", \"option2\": \"জওহরলাল নেহেরু\", \"option3\": \"সর্দার প্যাটেল\", \"option4\": \"ইন্দিরা গান্ধী\", \"target\": \"2\", \"subject\": \"history\", \"language\": \"Bengali\", \"split\": \"train\"}\n{\"question\": \"একটি অধিবর্ষে কত দিন থাকে?\", \"option1\": \"365\", \"option2\": \"364\", \"option3\": \"366\", \"option4\": \"360\", \"target\": \"3\", \"subject\": \"science\", \"language\": \"Bengali\", \"split\": \"train\"}\n{\"question\": \"பூமியில் உள்ள மிகப்பெரிய பெருங்கடல் எது?\", \"option1\": \"அட்லாண்டிக்\", \"option2\": \"இந்தியப் பெருங்கடல்\", \"option3\": \"ஆர்க்டிக்\", \"option4\": \"பசிபிக்\", \"target\": \"4\", \"subject\": \"geography\", \"language\": \"Tamil\", \"split\": \"train\"}\n{\"question\": \"பூமியில் எத்தனை கண்டங்கள் உள்ளன?\", \"option1\": \"5\", \"option2\": \"6\", \"option3\": \"7\", \"option4\": \"8\", \"target\": \"3\", \"subject\": \"geography\", \"language\": \"Tamil\", \"split\": \"train\"}\n{\"question\": \"தாவரங்கள் காற்றிலிருந்து எந்த வாயுவை உறிஞ்சுகின்றன?\", \"option1\": \"ஆக்ஸிஜன்\", \"option2\": \"நைட்ரஜன்\", \"option3\": \"கார்பன் டை ஆக்சைடு\", \"option4\": \"ஹைட்ரஜன்\", \"target\": \"3\", \"subject\": \"science\", \"language\": \"Tamil\", \"split\": \"train\"}\n{\"question\": \"இந்தியாவின் முதல் பிரதமர் யார்?\", \"option1\": \"மகாத்மா காந்தி\", \"option2\": \"ஜவஹர்லால் நேரு\", \"option3\": \"சர்தார் படேல்\", \"option4\": \"இந்திரா காந்தி\", \"target\": \"2\", \"subject\": \"history\", \"language\": \"Tamil\", \"split\": \"train\"}\n{\"question\": \"ஒரு லீப் ஆண்டில் எத்தனை நாட்கள் உள்ளன?\", \"option1\": \"365\", \"option2\": \"364\", \"option3\": \"366\", \"option4\": \"360\", \"target\": \"3\", \"subject\": \"science\", \"language\": \"Tamil\", \"split\": \"train\"}\n"}''')

for path, content in FILES.items():
    os.makedirs(os.path.dirname(path) or '.', exist_ok=True)
    with open(path, 'w', encoding='utf-8') as f:
        f.write(content)
print(f'wrote {len(FILES)} files')

## 4. Tests

These run on a tiny model and need no GPU. They cover the load-bearing
properties: the KV cache stays valid across an adapter swap, the incremental
decode path matches a full forward, the baseline adapter is exactly identity,
gradients isolate to the active adapter, and routed generation does not
collapse (a guard against adapters that destroy instruction-following and
thereby fake a token-efficiency win).

**Do not skip this cell.** It exercises the same PEFT/LoRA construction path
that training uses, on a tiny model, in seconds — so environment breakages
surface here rather than after a long model download.

In [ ]:
!python -m pytest tests/ -q

## 5. Train the 20 language experts

Plain causal-LM loss over in-language text — deliberately *not* answer-format
supervision, which would teach the adapters to skip the reasoning the eval is
measuring.

`lr=1e-5` is load-bearing: at `1e-4` for 60 steps the adapters destroy the
base model's instruction-following, and routed runs collapse to a bare
`Answer: C` in 3 tokens. That scores as a ~98% token saving but is just a
broken model.

Resumable — already-trained languages are skipped, so re-running after a
disconnect continues where it stopped.

In [ ]:
!python -m router.train_experts --steps 60 --n-train 48

## 6. Three-arm evaluation

| arm | routing | cache on switch |
|---|---|---|
| baseline | off | n/a |
| router | on | retained (cheap, approximate) |
| router_recomputed | on | rebuilt under the new expert (exact, re-pays prefill) |

The third arm exists because retaining the cache means the new expert attends
over a history a *different* expert encoded. Without it, a flat result is
uninterpretable — you cannot tell whether the experts are worthless or the
cache shortcut ate the gains.

Cost scales as `per_lang x 20 languages x 3 arms` generations, each up to 512
tokens. `--per-lang 5` is 300 generations (~20-40 min on a T4) and is enough to
see whether the numbers are sane. Raise it to 15-25 for a reportable result
once a small run has come back clean — below ~10 per language the accuracy
figures are mostly noise.

The `recomputed` arm is much slower than the other two: it re-prefills the
entire prefix on every expert switch, which is exactly the cost being
measured.

Progress prints to stderr every 10 items with a rate and ETA, so a long run is
distinguishable from a hang.

In [ ]:
!python pipeline.py --per-lang 5

## 7. Sweep the router to a plateau

Sweeps `check_every` x `window_chars` x `switch_patience` x `latin_min_conf`,
keeping the best and stopping once the score stops improving.

In [ ]:
!python optimize.py --per-lang 15 --patience 3

## 8. Results

In [ ]:
import json, os
for name in ['results/history.json', 'results/sweep.json']:
    if os.path.isfile(name):
        d = json.load(open(name))
        print('='*70); print(name)
        if name.endswith('history.json'):
            r = d[-1]
            for arm in ['baseline', 'router', 'router_recomputed']:
                if arm in r:
                    a = r[arm]
                    print(f"  {arm:20s} acc={a['accuracy']:.3f} "
                          f"gen={a['avg_gen_tokens']:.1f} proc={a['avg_total_processed']:.1f}")
            b, rt = r['baseline'], r['router']
            ex = r.get('router_recomputed', rt)
            print(f"  value of experts    (recomputed - baseline): {ex['accuracy']-b['accuracy']:+.3f}")
            print(f"  cost of shared cache (router - recomputed) : {rt['accuracy']-ex['accuracy']:+.3f}")
            print(f"  net                  (router - baseline)   : {rt['accuracy']-b['accuracy']:+.3f}")
            print(f"  prefill saved/item vs dispatch design      : {rt['avg_prefill_saved']:.1f} "
                  f"({rt['token_saving_ratio']*100:.1f}%)")
        else:
            print('  best config :', d['best'])
            print('  best score  :', d['best_score'])
            print('  candidates  :', len(d['history']))
    else:
        print(name, 'MISSING')

### Sanity check before believing any efficiency number

Two of the three bugs found while building this failed *toward* a flattering
token-efficiency result. Confirm the reasoning traces are intact — routed
generations should look like real CoT, not a bare answer line.

In [ ]:
import json
h = json.load(open('results/history.json'))[-1]
b, r = h['baseline']['avg_gen_tokens'], h['router']['avg_gen_tokens']
print(f'baseline {b:.1f} tokens vs routed {r:.1f} tokens  (ratio {r/b:.2f})')
print('OK — reasoning preserved' if r >= 0.4*b else
      'SUSPECT — routed generation collapsed; the "saving" is a broken model')

## 9. Package results for download

In [ ]:
!zip -qr results.zip results adapters && ls -lh results.zip
# Kaggle: appears under the notebook's Output tab.
# Colab: uncomment to download directly.
# from google.colab import files; files.download('results.zip')